# NYC Taxi Fare Prediction - Fase 1
Este notebook entrena un modelo de regresión para predecir el costo de un viaje en taxi en la ciudad de Nueva York.

In [ ]:

# === 1. IMPORTACIÓN DE LIBRERÍAS ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# === 2. CARGA DEL DATASET ===
df = pd.read_csv("train.csv", nrows=100000)  # Limitamos para velocidad

# === 3. EXPLORACIÓN BÁSICA ===
print(df.head())
print(df.info())
print(df.describe())

# === 4. LIMPIEZA BÁSICA ===
df = df.dropna()

# Eliminamos tarifas negativas o extremadamente altas
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 200)]

# Eliminamos coordenadas fuera de NYC
df = df[
    (df['pickup_latitude'].between(40, 42)) & (df['pickup_longitude'].between(-75, -72)) &
    (df['dropoff_latitude'].between(40, 42)) & (df['dropoff_longitude'].between(-75, -72))
]

# === 5. FEATURE ENGINEERING ===
# Convertimos datetime
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')
df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.day
df['weekday'] = df['pickup_datetime'].dt.weekday

# Distancia aproximada (distancia euclidiana)
def haversine(lat1, lon1, lat2, lon2):
    return np.sqrt((lat2 - lat1)**2 + (lon2 - lon1)**2)

df['distance'] = haversine(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)

# === 6. SELECCIÓN DE VARIABLES ===
features = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'hour', 'weekday', 'distance']
target = 'fare_amount'

X = df[features]
y = df[target]

# === 7. ENTRENAMIENTO DEL MODELO ===
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# === 8. EVALUACIÓN ===
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"RMSE: {rmse:.2f}")

# === 9. GUARDAR MODELO ENTRENADO ===
joblib.dump(model, 'model.pkl')
print("Modelo guardado como model.pkl")

# === 10. HACER UNA PREDICCIÓN DE PRUEBA ===
ejemplo = X_val.iloc[0:1]
prediccion = model.predict(ejemplo)
print(f"Predicción: {prediccion[0]:.2f} | Valor real: {y_val.iloc[0]:.2f}")
